# Model Evaluation

In [22]:
from pyspark.sql import SparkSession

In [23]:
spark = (
    SparkSession.builder
    .appName("BusServiceReliability")
    .master("local[*]")
    .getOrCreate()
)

In [24]:
from pyspark.ml.classification import DecisionTreeClassificationModel

model = DecisionTreeClassificationModel.load(
    "../models/decision_tree_model"
)

In [25]:
from pyspark.ml.classification import LogisticRegressionModel, RandomForestClassificationModel

lr_model = LogisticRegressionModel.load("logistic_regression_model")
rf_model = RandomForestClassificationModel.load("random_forest_model")

In [26]:
lr_predictions = lr_model.transform(test_data)
rf_predictions = rf_model.transform(test_data)

In [27]:
df = spark.read.parquet("../outputs/cleaned_timetable_parquet")

## Prepare Dataset

In [28]:
from pyspark.sql.functions import when, col, countDistinct, stddev
from pyspark.ml.feature import StringIndexer, VectorAssembler

# 1. Same target variable
df = df.withColumn(
    "target",
    when(col("route_size") == "Long", 1).otherwise(0)
)

# 2. Same indexer
indexer = StringIndexer(inputCol="service_code", outputCol="service_code_index")
df = indexer.fit(df).transform(df)

# 3. Same derived features as training
service_features = df.groupBy("service_code").agg(
    countDistinct("journey_id").alias("num_journeys"),
    stddev("stop_sequence").alias("stop_seq_std")
)
df = df.join(service_features, on="service_code", how="left")

# 4. Same VectorAssembler inputs as training
assembler = VectorAssembler(
    inputCols=["service_code_index", "num_journeys", "stop_seq_std"],
    outputCol="features"
)
dataset = assembler.transform(df)

# 5. Same split, same seed
train_data, test_data = dataset.randomSplit([0.8, 0.2], seed=42)

## Generate Predictions

In [29]:
predictions = model.transform(test_data)

## Model Accuracy

In [30]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
import pandas as pd

# 1. Set up all Evaluators
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction", metricName="accuracy")
evaluator_precision = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction", metricName="weightedPrecision")
evaluator_recall = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction", metricName="f1")
evaluator_roc = BinaryClassificationEvaluator(labelCol="target", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# 2. Calculate metrics for the Decision Tree predictions
dt_metrics = {
    "Model": "Decision Tree",
    "Accuracy": evaluator_accuracy.evaluate(predictions),
    "Precision": evaluator_precision.evaluate(predictions),
    "Recall": evaluator_recall.evaluate(predictions),
    "F1-score": evaluator_f1.evaluate(predictions),
    "ROC-AUC": evaluator_roc.evaluate(predictions)
}

# 3. Convert to Pandas DataFrame for a nice table
metrics_table = [dt_metrics]
results_df = pd.DataFrame(metrics_table)

# Format numbers to 4 decimal places
for col in ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]:
    results_df[col] = results_df[col].round(4)

print("\nDECISION TREE EVALUATION METRICS \n")
display(results_df)

[Stage 344:================================>                       (7 + 5) / 12]


DECISION TREE EVALUATION METRICS 



,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Decision Tree,0.8511,0.7438,0.8511,0.7932,0.4937


In [31]:
def get_metrics(preds, model_name):
    return {
        "Model": model_name,
        "Accuracy": evaluator_accuracy.evaluate(preds),
        "Precision": evaluator_precision.evaluate(preds),
        "Recall": evaluator_recall.evaluate(preds),
        "F1-score": evaluator_f1.evaluate(preds),
        "ROC-AUC": evaluator_roc.evaluate(preds)
    }

all_metrics = [
    get_metrics(lr_predictions, "Logistic Regression"),
    get_metrics(rf_predictions, "Random Forest"),
    get_metrics(predictions, "Decision Tree")
]

comparison_df = pd.DataFrame(all_metrics)
for c in ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]:
    comparison_df[c] = comparison_df[c].round(4)

print("\n MODEL COMPARISON — ALL 3 CLASSIFIERS \n")
display(comparison_df)


 MODEL COMPARISON — ALL 3 CLASSIFIERS 



,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression,0.8576,0.8224,0.8576,0.8303,0.8464
1,Random Forest,0.8624,0.7438,0.8624,0.7987,0.5000
2,Decision Tree,0.8511,0.7438,0.8511,0.7932,0.4937


## Display Predictions

In [32]:
predictions.select(
    "route_size",
    "target",
    "prediction",
    "probability"
).show(10, truncate=False)

+----------+------+----------+-----------+
|route_size|target|prediction|probability|
+----------+------+----------+-----------+
|Short     |0     |0.0       |[1.0,0.0]  |
|Short     |0     |0.0       |[1.0,0.0]  |
|Short     |0     |0.0       |[1.0,0.0]  |
|Short     |0     |0.0       |[1.0,0.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
|Short     |0     |0.0       |[1.0,0.0]  |
|Short     |0     |0.0       |[1.0,0.0]  |
+----------+------+----------+-----------+
only showing top 10 rows


## Prediction Distribution

In [33]:
predictions.groupBy("prediction").count().show()

[Stage 515:==================>                                     (4 + 8) / 12]

+----------+-----+
|prediction|count|
+----------+-----+
|       0.0|53293|
|       1.0|  626|
+----------+-----+



## Summary

The Decision Tree model was successfully evaluated using the testing dataset. The model generated predictions for unseen data and achieved a measurable classification accuracy. The evaluation results indicate that the model can distinguish between long and non-long routes based on the selected features. These findings demonstrate the effectiveness of Spark MLlib for building scalable machine learning pipelines on timetable data.